# `nanoddpm-ddim`

>**Goal:** _Fast Diffusion from scratch._

Upgrading `nanoddpm` with a **Mini-UNet, Classifier-Free Guidance (CFG), Denoising Diffusion Implicit Models (DDIM) sampling, and PCA-FID evaluation.** It is run on the **CIFAR-10** dataset. No high-level wrappers, no hidden abstractions, just raw **PyTorch** + the core diffusion equations.
<br><br>

---
## 1. Classifier-Free Guidance (CFG)
Standard conditional diffusion models require a separate classifier to steer generation. CFG removes that dependency by training a single model on **paired conditional/unconditional data**.

During training, we randomly mask the class label (replace with `-1` or `∅`) ~10% of the time. This forces the network to learn two representations:
- `ε_θ(x_t, t, y)` : noise prediction with class conditioning
- `ε_θ(x_t, t, ∅)` : noise prediction without conditioning

At inference, we combine them linearly:
$$
\hat{\epsilon} = \epsilon_\theta(x_t, t, \emptyset) + w \cdot \left( \epsilon_\theta(x_t, t, y) - \epsilon_\theta(x_t, t, \emptyset) \right)
$$
Where `w` (`cfg_scale`) controls how strongly we push toward the target class.
- `w = 1.0` → standard conditional generation
- `w = 3.0–7.5` → sharper, more class-aligned outputs (at the cost of diversity)
- `w > 7.5` → artifacts & oversaturation

**Why it works:** The difference term `(ε_cond - ε_uncond)` points in the direction of the data manifold for class `y`. Scaling it amplifies the "pull" toward that class without needing gradients from an external network.
<br><br>

---
## 2. DDIM: Deterministic Fast Sampling
DDPM sampling is slow because it adds random noise at every reverse step. DDIM reparameterizes the reverse process as a **deterministic ODE**, allowing us to skip steps safely.

Starting from the DDPM reverse mean:
$$
\mu_\theta(x_t, t) = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}} \epsilon_\theta(x_t, t) \right)
$$

We can predict the original image $x_0$ directly:
$$
x_0 = \frac{x_t - \sqrt{1-\bar{\alpha}_t} \epsilon_\theta}{\sqrt{\bar{\alpha}_t}}
$$

DDIM replaces the stochastic update with:
$$
x_{t-1} = \sqrt{\bar{\alpha}_{t-1}} x_0 + \sqrt{1 - \bar{\alpha}_{t-1} - \sigma_t^2} \epsilon_\theta + \sigma_t z
$$
Setting $\sigma_t = 0$ removes the random noise `z`, making the path deterministic. Because the trajectory is fixed, we can evaluate the network at a subset of steps (e.g., `t = 500, 250, 100, 50, 0`) and interpolate cleanly.

**Result:** 10–50 steps instead of 500–1000. Same weights, same training loop, just a swapped sampler.
<br><br>

---
## 3. PCA-FID: Lightweight Quality Tracking
**Fréchet Inception Distance (FID)** is a statistical metric used to evaluate the quality and diversity of images produced by generative models like _DDPM_. It measures the distance between the distribution of generated images and real training images.

Standard FID uses Inception-V3 to extract 2048-d features. That's heavy, opaque, and overkill for CIFAR-10. PCA-FID replaces the black-box extractor with **unsupervised dimensionality reduction**:

1. Flatten images to `[B, 3072]`
2. Compute top-32 principal components across real + generated batches
3. Project both sets into the 32D subspace
4. Run the standard FID formula on the projected means/covariances

$$
\text{PCA-FID} = \|\mu_r - \mu_g\|^2 + \text{Tr}(\Sigma_r + \Sigma_g - 2\sqrt{\Sigma_r \Sigma_g})
$$


---


## Imports & Config

In [ ]:
import torch, torch.nn as nn, torch.optim as optim, torchvision, torchvision.transforms as T
import matplotlib.pyplot as plt, numpy as np, math, json, torch.nn.functional as F
from tqdm import trange, tqdm
import ipywidgets as widgets
import time

import base64
from IPython.display import Image, display

%matplotlib inline
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === CONFIG (edit these or use Cell 8 sliders later) ===
EPOCHS = 50
BATCH_SIZE = 128
T_STEPS = 1000
CFG_SCALE = 4.0
RESIZE = 32
print(f"▶ Device: {device} | Steps: {T_STEPS} | Resize: {RESIZE}")

## Noise Schedule & Forward Diffusion

In [ ]:
beta = torch.linspace(1e-4, 0.02, T_STEPS, device=device)
alpha = 1.0 - beta
alpha_bar = torch.cumprod(alpha, dim=0)

def forward_diffusion(x0, t):
    """q(x_t | x_0) = sqrt(ᾱ_t)·x_0 + sqrt(1-ᾱ_t)·ε"""
    sqrt_ab = torch.sqrt(alpha_bar[t])[:, None, None, None]
    sqrt_1m = torch.sqrt(1.0 - alpha_bar[t])[:, None, None, None]
    eps = torch.randn_like(x0, device=device)
    return sqrt_ab * x0 + sqrt_1m * eps, eps

## Dataset Loading & Preparation

In [ ]:
transform = T.Compose([T.Resize(RESIZE), T.ToTensor(), T.Normalize([0.5]*3, [0.5]*3)])
dataset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
real_batch, real_labels = next(iter(torch.utils.data.DataLoader(dataset, batch_size=256, shuffle=False)))
real_batch = real_batch.to(device)
print(f"▶ CIFAR-10 loaded. Shape: {dataset[0][0].shape}")

## Mini-UNet Model

In [ ]:
_res_block = """
graph TD
    A[Input x] --> B[Conv2D 3x3]
    B --> C[GroupNorm]
    C --> D[SiLU]

    E[Time t] --> F[Sinusoidal Emb]
    F --> G[MLP: SiLU + Linear]
    G --> H((+))

    I[Class y] --> J[Embedding]
    J --> K[Dropout 0.1]
    K --> H

    H --> L[Add to Feature Map]
    D --> L

    L --> M[Conv2D 3x3]
    M --> N[GroupNorm]
    N --> O[SiLU]

    A --> P[Skip: Conv 1x1 or Identity]
    P --> Q((+))
    O --> Q
    Q --> R[Output]
"""

_mini_UNET = """
graph TD
    A[Input: 3xHxW] --> D1[ResBlock: 3→ch]
    D1 -->|Skip d1| S1[(d1)]

    D1 --> P1[AvgPool 2x2]
    P1 --> D2[ResBlock: ch→ch*2]
    D2 -->|Skip d2| S2[(d2)]

    D2 --> P2[AvgPool 2x2]
    P2 --> M[ResBlock: ch*2→ch*2]

    M --> U1[Int: 2x Nearest]
    S2 --> C1[Concat]
    U1 --> C1
    C1 --> U1B[ResBlock: ch*3→ch]

    U1B --> U2[Int: 2x Nearest]
    S1 --> C2[Concat]
    U2 --> C2
    C2 --> U2B[ResBlock: ch*2→3]

    U2B --> OUT[Conv2D 1x1]
    OUT --> Z[Output: 3xHxW]
"""

In [ ]:
def render_mermaid(graph_code):
    graphbytes = graph_code.encode("utf-8")
    base64_bytes = base64.b64encode(graphbytes)
    base64_string = base64_bytes.decode("utf-8")
    display(Image(url="https://mermaid.ink/img/" + base64_string))

def render_mermaid_HW(graph_code, width=None, height=None):
    # Fix: Encode and decode using 'utf-8' to handle Unicode characters
    graphbytes = graph_code.encode("utf-8")
    base64_bytes = base64.b64encode(graphbytes)
    base64_string = base64_bytes.decode("utf-8")
    display(Image(url="https://mermaid.ink/img/" + base64_string, width=width, height=height))


In [ ]:
render_mermaid_HW(_res_block, width=600, height=600)

In [ ]:
render_mermaid_HW(_mini_UNET, width=300, height=860)

In [ ]:
def sinusoidal_embedding(t, dim, max_period=10000):
    half = dim // 2
    freqs = torch.exp(-math.log(max_period) * torch.arange(half, dtype=torch.float32, device=t.device) / half)
    args = t[:, None].float() * freqs[None, :]
    return torch.cat([torch.cos(args), torch.sin(args)], dim=1)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim, num_classes=10):
        super().__init__()
        # Dynamically set num_groups for GroupNorm
        # If out_ch is 3, set num_groups to 3. Otherwise, use 8.
        num_groups_gn = min(8, out_ch)

        self.conv1, self.norm1 = nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.GroupNorm(num_groups_gn, out_ch)
        self.conv2, self.norm2 = nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.GroupNorm(num_groups_gn, out_ch)
        self.time_mlp = nn.Sequential(nn.SiLU(), nn.Linear(time_dim, out_ch))
        self.class_emb = nn.Embedding(num_classes, time_dim)
        self.dropout = nn.Dropout(0.1)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.time_dim = time_dim # Store time_dim

    def forward(self, x, t, labels=None):
        h = F.silu(self.norm1(self.conv1(x)))
        # Fix: Pass self.time_dim to sinusoidal_embedding
        t_emb = self.time_mlp(sinusoidal_embedding(t, self.time_dim))
        c_emb = torch.zeros_like(t_emb) if labels is None else self.class_emb(labels)
        h = h + (t_emb + self.dropout(c_emb))[:, :, None, None]
        return F.silu(self.norm2(self.conv2(h))) + self.skip(x)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim, num_classes=10, dropout=0.1):
        super().__init__()

        # If out_ch is 3, set num_groups to 3. Otherwise, use 8.
        num_groups_gn = min(8, out_ch)

        self.time_dim = time_dim # Store time_dim
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm1 = nn.GroupNorm(num_groups_gn, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(num_groups_gn, out_ch)
        self.time_mlp = nn.Sequential(nn.SiLU(), nn.Linear(time_dim, out_ch))
        self.class_emb = nn.Embedding(num_classes, time_dim)
        self.class_proj = nn.Linear(time_dim, out_ch)  # Projects class emb to match out_ch
        self.dropout = nn.Dropout(dropout)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t, labels=None):
        h = F.silu(self.norm1(self.conv1(x)))
        t_emb = self.time_mlp(sinusoidal_embedding(t, self.time_dim))

        # Handle CFG: zero embedding for unconditional, projected + dropout for conditional
        if labels is None:
            c_emb = torch.zeros_like(t_emb)
        else:
            c_emb = self.class_proj(self.dropout(self.class_emb(labels)))

        h = h + (t_emb + c_emb)[:, :, None, None]
        return F.silu(self.norm2(self.conv2(h))) + self.skip(x)

class MiniUNet(nn.Module):
    def __init__(self, time_dim=128, ch=32):
        super().__init__()
        self.down1 = ResBlock(3, ch, time_dim)
        self.down2 = ResBlock(ch, ch*2, time_dim)
        self.mid = ResBlock(ch*2, ch*2, time_dim)
        self.up1 = ResBlock(ch*4, ch, time_dim)
        self.up2 = ResBlock(ch*2, 3, time_dim)
        self.out = nn.Conv2d(3, 3, 1)

    def forward(self, x, t, labels=None):
        d1 = self.down1(x, t, labels)
        d2 = self.down2(F.avg_pool2d(d1, 2), t, labels)
        x = self.mid(F.avg_pool2d(d2, 2), t, labels)
        x = self.up1(torch.cat([F.interpolate(x, scale_factor=2), d2], 1), t, labels)
        x = self.up2(torch.cat([F.interpolate(x, scale_factor=2), d1], 1), t, labels)
        return self.out(x)

model = MiniUNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=2e-3)
print(f"▶ Params: {sum(p.numel() for p in model.parameters()):,}")

## PCA-FID Function

In [ ]:
def pca_fid(real, gen, n_components=32):
    r, g = real.view(real.shape[0], -1).cpu().double(), gen.view(gen.shape[0], -1).cpu().double()
    data = torch.cat([r, g], 0)
    mean = data.mean(0, keepdim=True)
    U, S, V = torch.pca_lowrank(data - mean, q=n_components)
    proj = (data - mean) @ V
    r_p, g_p = proj[:real.shape[0]], proj[real.shape[0]:]
    mu_r, mu_g = r_p.mean(0), g_p.mean(0)
    var_r, var_g = r_p.var(0)+1e-6, g_p.var(0)+1e-6
    return ((mu_r-mu_g)**2).sum().item() + (var_r+var_g-2*torch.sqrt(var_r*var_g)).sum().item()

## Model Training

In [ ]:
metrics_log = []
for epoch in trange(1, EPOCHS+1, desc="Training"):
    model.train()
    epoch_loss, count = 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        mask = torch.rand_like(labels.float()) < 0.1
        train_labels = labels.clone()
        train_labels[mask] = -1 # Labels can be 0-9 or -1

        t = torch.randint(0, T_STEPS, (imgs.shape[0],), device=device)
        xt, eps = forward_diffusion(imgs, t)

        optimizer.zero_grad()

        # Split batch into conditional and unconditional for training
        conditional_indices = (train_labels != -1).nonzero(as_tuple=True)[0]
        unconditional_indices = (train_labels == -1).nonzero(as_tuple=True)[0]

        pred_eps = torch.zeros_like(eps)

        if len(conditional_indices) > 0:
            pred_eps[conditional_indices] = model(
                xt[conditional_indices],
                t[conditional_indices],
                train_labels[conditional_indices]
            )

        if len(unconditional_indices) > 0:
            pred_eps[unconditional_indices] = model(
                xt[unconditional_indices],
                t[unconditional_indices],
                None # Pass None for labels to trigger the zero embedding path
            )

        loss = F.mse_loss(pred_eps, eps)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()*imgs.shape[0]
        count += imgs.shape[0]

    # Quick eval after each epoch
    model.eval()
    with torch.no_grad():
        gen = torch.randn(128, 3, RESIZE, RESIZE, device=device)
        # Use real_labels for conditional generation during evaluation
        eval_cond_labels = real_labels[:128].to(device)

        for t in reversed(range(0, T_STEPS, 50)):
            t_t = torch.full((128,), t, device=device)

            # For eps_u (unconditional), pass None for labels
            eps_u = model(gen, t_t, None)
            # For eps_c (conditional), pass the specific class labels
            eps_c = model(gen, t_t, eval_cond_labels)
            eps = eps_u + CFG_SCALE * (eps_c - eps_u)

            a, ab = alpha[t], alpha_bar[t]
            pred_x0 = (gen - torch.sqrt(1-ab)*eps)/torch.sqrt(ab)
            a_prev = alpha[t-50] if t>=50 else alpha[0]
            gen = torch.sqrt(a_prev)*pred_x0 + torch.sqrt(1-a_prev)*eps
            gen = torch.clip(gen, -1, 1)

        # fid = ((real_batch[:128].view(128,-1).mean(0) - gen.view(128,-1).mean(0))**2).sum().item()
        fid = pca_fid(real_batch[:128], gen)

    metrics_log.append({'epoch':epoch, 'loss':epoch_loss/count, 'pca_fid':fid})
    print(f"  Epoch {epoch:02d} | Loss: {metrics_log[-1]['loss']:.4f} | PCA-FID: {fid:.1f}")


with open('nanoddpm_pro_metrics.json','w') as f:
  json.dump(metrics_log, f)
print("Metrics saved to nanoddpm_pro_metrics.json")

## Visualization

In [ ]:
def plot_results():
    fig, axs = plt.subplots(1,2,figsize=(10,3))
    axs[0].plot([m['epoch'] for m in metrics_log], [m['loss'] for m in metrics_log], marker='o')
    axs[0].set_title('Loss')
    axs[0].grid(0.3)
    axs[1].plot([m['epoch'] for m in metrics_log], [m['pca_fid'] for m in metrics_log], marker='s', color='orange')
    axs[1].set_title('PCA-FID (↓)')
    axs[1].grid(0.3)
    plt.tight_layout()
    plt.show()

plot_results()

In [ ]:
@torch.no_grad()
def sample_trajectory(n=9, steps_to_plot=[999, 998, 997, 996, 995, 0], cls_idx=5, cfg_scale=4.0):
    """Visualize the denoising trajectory from noise (t=999) to clean image (t=0)"""
    model.eval()
    x = torch.randn(n, 3, RESIZE, RESIZE, device=device)
    labels = torch.full((n,), cls_idx, device=device)
    trajectory = []

    # Use DDIM steps (deterministic)
    for t in reversed(range(T_STEPS)):
        t_t = torch.full((n,), t, device=device)

        # CFG-guided prediction
        eps_u = model(x, t_t, None)
        eps_c = model(x, t_t, labels)
        eps = eps_u + cfg_scale * (eps_c - eps_u)

        # DDIM reverse step
        a, ab = alpha[t], alpha_bar[t]
        pred_x0 = (x - torch.sqrt(1 - ab) * eps) / torch.sqrt(ab)
        a_prev = alpha[t-1] if t > 0 else alpha[0]
        x = torch.sqrt(a_prev) * pred_x0 + torch.sqrt(1 - a_prev) * eps
        x = torch.clip(x, -1.0, 1.0)

        trajectory.append(x.clone().cpu())

    # Plot trajectory
    fig, axes = plt.subplots(1, len(steps_to_plot), figsize=(15, 2))
    for i, t_idx in enumerate(steps_to_plot):
        idx = T_STEPS - 1 - t_idx  # Reverse indexing
        if idx < len(trajectory):
            grid = torchvision.utils.make_grid(trajectory[idx], nrow=3, normalize=True, value_range=(-1, 1))
            axes[i].imshow(grid.permute(1, 2, 0).numpy())
            # axes[i].set_title(f"t={t_idx}")
            axes[i].set_title(f"t={idx+1 if idx != 0 and idx != 999 else idx}")
            axes[i].axis('off')
    plt.tight_layout()
    plt.show()

# Example: Generate trajectory for class 5 (dogs)
# sample_trajectory(n=9, steps_to_plot=[999, 900, 800, 700, 600, 500, 400, 300, 200, 100, 0], cls_idx=5, cfg_scale=4.0)
sample_trajectory(n=9, steps_to_plot=[999, 998, 997, 996, 995, 0], cls_idx=5, cfg_scale=4.0)

In [ ]:
@torch.no_grad()
def generate(cfg, steps, cls_idx):
    """
    Generate images with DDIM sampling
    Args:
    - cfg (float): CFG scale
    - steps (int): Number of DDIM steps
    - cls_idx (int): Class index for conditional generation
    """
    try:
        model.eval()
        n = 9

        # Ensure inputs are correct types (widgets sometimes pass floats as strings)
        cfg = float(cfg)
        steps = int(steps)
        cls_idx = int(cls_idx)

        x = torch.randn(n, 3, RESIZE, RESIZE, device=device)
        labels = torch.full((n,), cls_idx, dtype=torch.long, device=device)
        step_size = max(1, T_STEPS // steps)

        for t in reversed(range(0, T_STEPS, step_size)):
            t_t = torch.full((n,), t, dtype=torch.long, device=device)

            # Unconditional: pass None (not -1) to avoid embedding crash
            eps_u = model(x, t_t, None)
            eps_c = model(x, t_t, labels)
            eps = eps_u + cfg * (eps_c - eps_u)

            a, ab = alpha[t], alpha_bar[t]
            pred_x0 = (x - torch.sqrt(1 - ab) * eps) / torch.sqrt(ab)
            a_prev = alpha[t - step_size] if t >= step_size else alpha[0]
            x = torch.sqrt(a_prev) * pred_x0 + torch.sqrt(1 - a_prev) * eps
            x = torch.clip(x, -1.0, 1.0)

        # Move to CPU for plotting
        grid = torchvision.utils.make_grid(x, nrow=3, normalize=True, value_range=(-1, 1))
        plt.figure(figsize=(4, 4))
        plt.imshow(grid.cpu().permute(1, 2, 0).numpy())
        plt.axis('off')
        plt.show()

    except Exception as e:
        print(f"Generation error: {e}")
        print("Tip: Ensure model is trained and all globals (alpha, T_STEPS, etc.) are defined.")

# Interactive widget
widgets.interact(
    generate,
    cfg=widgets.FloatSlider(value=4.0, min=1.0, max=7.5, step=0.5, description='CFG Scale'),
    steps=widgets.IntSlider(value=20, min=10, max=100, step=5, description='DDIM Steps'),
    cls_idx=widgets.IntSlider(value=0, min=0, max=9, step=1, description='Class (0-9)')
)